# Component 3 — Demand forecast: when does ML help, and what does it mean for stock?

`inventory_weekly.ipynb` showed that the weekly model beats naive forecasts overall (MAE 21.7 % lower)
and that the original monthly R² 0.931 was inflated by target leakage (0.889 without it).
This notebook asks two follow-up questions:

1. **When does ML help?** Shop demand jumps around Sinhala and Tamil New Year (Avurudu). A naive
   forecast ("same as last week") only reacts *after* the jump. Is ML's advantage bigger in festival
   weeks and in the weeks right after?
2. **What does it mean for stock?** If a shop stocks exactly the forecast each week, how many units
   would it be **short** (lost sales) and how many **left over** (overstock, spoilage risk)?

Same data, time split, 4-week gap and champion model as `inventory_weekly.ipynb`. The test period
(2025-07 → 2026-06) is not used to choose anything. The original notebooks, CSV files and `.pkl`
files are not changed.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## 1. Same data, split and champion (RandomForest) as `inventory_weekly.ipynb`

In [ ]:
df = pd.read_csv('demand_forecast_weekly.csv').drop_duplicates()
df['week_start'] = pd.to_datetime(df['iso_year'].astype(str) + '-W' + df['iso_week'].astype(str).str.zfill(2) + '-1',
                                  format='%G-W%V-%u')
df = df.sort_values(['item', 'week_start']).reset_index(drop=True)

X = df.drop(columns=['target_units_sold', 'week_start'])
y = df['target_units_sold'].astype(float)
num_cols = [c for c in X.columns if c not in ('item', 'category')]

GAP = pd.Timedelta(weeks=4)
TEST_START = pd.Timestamp('2025-07-01')
dev = (df['week_start'] < TEST_START - GAP).values
test = (df['week_start'] >= TEST_START).values

pre = ColumnTransformer([
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]),
     ['item', 'category']),
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
])
model = Pipeline([('pre', pre), ('m', RandomForestRegressor(n_estimators=300, max_depth=12, n_jobs=-1,
                                                            random_state=RANDOM_STATE))]).fit(X[dev], y[dev])

T = df[test].copy()
T['ML'] = model.predict(X[test])
T['Naive: last week'] = T['lag1_units']
T['Naive: 4-week mean'] = T['rolling4_mean_units']
METHODS = ['ML', 'Naive: last week', 'Naive: 4-week mean']
print(f"Test: {len(T)} rows, {T['item'].nunique()} items, {T['week_start'].min().date()} -> {T['week_start'].max().date()}")

## 2. Label each test week: festival, after festival, normal

`festival_season = 1` marks the Avurudu weeks. The two weeks after the last festival week are labelled
*after festival* — the time when demand drops back and "same as last week" is badly wrong.

In [ ]:
fest_weeks = sorted(T.loc[T['festival_season'] == 1, 'week_start'].unique())
after_weeks = [fest_weeks[-1] + pd.Timedelta(weeks=k) for k in (1, 2)]
T['period'] = np.select([T['week_start'].isin(fest_weeks), T['week_start'].isin(after_weeks)],
                        ['festival', 'after festival'], 'normal')
print('Festival weeks      :', [str(w.date()) for w in fest_weeks])
print('After-festival weeks:', [str(w.date()) for w in after_weeks])
print(T.groupby('period').agg(rows=('target_units_sold', 'size'), mean_units=('target_units_sold', 'mean')).round(1))

## 3. Experiment 1 — forecast error by period

MAE = average number of units the forecast is off by, per item per week (lower is better).

In [ ]:
def mae(a, b): return float(np.mean(np.abs(a - b)))

rows = []
for period in ['normal', 'festival', 'after festival', 'ALL']:
    part = T if period == 'ALL' else T[T['period'] == period]
    row = {'period': period, 'rows': len(part)}
    for m in METHODS:
        row[f'MAE {m}'] = mae(part['target_units_sold'], part[m])
    best_naive = min(row['MAE Naive: last week'], row['MAE Naive: 4-week mean'])
    row['ML better than best naive %'] = (1 - row['MAE ML'] / best_naive) * 100
    rows.append(row)
by_period = pd.DataFrame(rows).set_index('period').round(2)
by_period

### Is ML's advantage real in each period? Bootstrap 95 % confidence intervals

Rows are resampled within each period (1,000 times). The interval is for *MAE of best naive − MAE of ML*:
above 0 means ML is better. The festival and after-festival periods have only a few weeks, so their
intervals are wide — that is a limitation, stated in the conclusion.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
for period in ['normal', 'festival', 'after festival', 'ALL']:
    part = T if period == 'ALL' else T[T['period'] == period]
    a = part['target_units_sold'].to_numpy()
    err = {m: np.abs(a - part[m].to_numpy()) for m in METHODS}
    naive = 'Naive: last week' if err['Naive: last week'].mean() < err['Naive: 4-week mean'].mean() else 'Naive: 4-week mean'
    idx = rng.integers(0, len(a), size=(1000, len(a)))
    gain = err[naive][idx].mean(1) - err['ML'][idx].mean(1)
    lo, hi = np.percentile(gain, [2.5, 97.5])
    print(f'{period:15s} best naive = {naive:18s} | ML better by {gain.mean():5.2f} units/week  95% CI [{lo:5.2f}, {hi:5.2f}]')

## 4. Experiment 2 — what it means for stock

Simple policy: each week the shop has exactly the forecast quantity on the shelf.

- **Short** = units customers wanted but the shop did not have (lost sales) = max(0, actual − forecast).
- **Left over** = units bought but not sold that week = max(0, forecast − actual) — money tied up,
  and for vegetables and fruit, spoilage.

In [ ]:
actual = T['target_units_sold']
stock_rows = []
for m in METHODS:
    short = np.maximum(0, actual - T[m]); over = np.maximum(0, T[m] - actual)
    for period in ['festival', 'after festival', 'ALL']:
        mask = np.ones(len(T), bool) if period == 'ALL' else (T['period'] == period).to_numpy()
        stock_rows.append({'method': m, 'period': period,
                           'short units': short[mask].sum(), 'short % of demand': short[mask].sum() / actual[mask].sum() * 100,
                           'left-over units': over[mask].sum(), 'left-over % of demand': over[mask].sum() / actual[mask].sum() * 100})
stock = pd.DataFrame(stock_rows).set_index(['period', 'method']).sort_index().round(1)
stock

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
periods = ['normal', 'festival', 'after festival']
x = np.arange(len(periods)); w = 0.27
colors = {'ML': '#1f5fa8', 'Naive: last week': '#bbb', 'Naive: 4-week mean': '#777'}
for i, m in enumerate(METHODS):
    ax[0].bar(x + (i - 1) * w, by_period.loc[periods, f'MAE {m}'], w, label=m, color=colors[m])
ax[0].set_xticks(x); ax[0].set_xticklabels(periods); ax[0].set_ylabel('MAE (units / item / week)')
ax[0].set_title('Forecast error by period (test 2025-07 to 2026-06)'); ax[0].legend()

s = stock.xs('ALL', level='period').loc[METHODS]
ax[1].barh(METHODS, s['short % of demand'], color='#c0392b', label='Short (lost sales)')
ax[1].barh(METHODS, s['left-over % of demand'], left=s['short % of demand'], color='#e0a96d', label='Left over')
ax[1].invert_yaxis(); ax[1].set_xlabel('% of total demand in the test period')
ax[1].set_title('Stocking exactly the forecast'); ax[1].legend(loc='upper right')
plt.tight_layout(); plt.savefig('component3_festival_stock.png', dpi=120); plt.show()

## 5. Conclusion

*Written after reading the numbers above — fill in with the actual results:*

- Overall, ML is ___ % better than the best naive forecast (interval ___).
- In festival weeks: ___ ; in the weeks after the festival: ___.
- Stocking the ML forecast: short ___ % and left over ___ % of demand, compared with ___ for "same as last week".

**Limitations.** The test period contains only one Avurudu (3 festival weeks, 2 after-festival weeks),
so the festival results are indicative, not conclusive. The data is a public-price-based weekly series,
not one real shop's sales. The stock policy ignores safety stock and lead time (see
`inventory_weekly.ipynb`, section 7 for the reorder level used in the app).